In [1]:

import pandas as pd
import numpy as np
import os
import openpyxl
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu, ttest_ind

# Load data

In [2]:
input_dir = "/mnt/c/users/helen/Desktop/FIBERS"

In [3]:
dfs = []

for root, dirs, files in os.walk(input_dir):
    for filename in files:
        if filename.lower().endswith(".csv"):

            path = os.path.join(root, filename)

            df = pd.read_csv(path)

            # Optional metadata
            df["File"] = os.path.splitext(filename)[0].replace(" ", "_")
            df["Path"] = path

            dfs.append(df) 

# Combine all tables
data = pd.concat(dfs, ignore_index=True)

# Create ROI column
data['ROI'] = data['Label'].apply(lambda x: x.split(":")[1])
data['Sample_name'] = data['File'].apply(lambda x: x.split("_")[1].split("-")[0])

# Delete first 3 columns
data.drop(data.columns[[0, 1, 2]], axis=1, inplace=True)

# Reorder columns
data = data[
    [
        "Sample_name",
        "File",
        "Measurement_type",
        "Length",
        "ROI",
        "Path"
    ]
]

In [4]:
# Fix Sample name manually for some dataframe
data["Sample_name"] = data["Sample_name"].apply(
    lambda x: "WT" if "1h15min" in x else x
)

In [5]:
# Split data into 2 dataframes
speed = data[data['Measurement_type']=='Fiber_length']
iod = data[data['Measurement_type']=='Interorigin_distance']

In [6]:
iod.head()

,Sample_name,File,Measurement_type,Length,ROI,Path
18,MGS1,siORC1_MGS1-04_Interorigin_distance,Interorigin_distance,125.399,0550-0404,/mnt/c/users/helen/Desktop/FIBERS/010726/siORC...
19,MGS1,siORC1_MGS1-04_Interorigin_distance,Interorigin_distance,171.724,0477-0521,/mnt/c/users/helen/Desktop/FIBERS/010726/siORC...
20,MGS1,siORC1_MGS1-04_Interorigin_distance,Interorigin_distance,97.546,0646-0786,/mnt/c/users/helen/Desktop/FIBERS/010726/siORC...
27,MGS1,siORC1_MGS1-07_Interorigin_distance,Interorigin_distance,106.301,0493-0678,/mnt/c/users/helen/Desktop/FIBERS/010726/siORC...
28,MGS1,siORC1_MGS1-07_Interorigin_distance,Interorigin_distance,232.019,0814-0659,/mnt/c/users/helen/Desktop/FIBERS/010726/siORC...


In [7]:
speed.head()

,Sample_name,File,Measurement_type,Length,ROI,Path
0,MGS1,siORC1_MGS1-02_Fiber_length,Fiber_length,51.108,0303-0404,/mnt/c/users/helen/Desktop/FIBERS/010726/siORC...
1,MGS1,siORC1_MGS1-02_Fiber_length,Fiber_length,37.202,0273-0360,/mnt/c/users/helen/Desktop/FIBERS/010726/siORC...
2,MGS1,siORC1_MGS1-03_Fiber_length,Fiber_length,55.027,0944-0253,/mnt/c/users/helen/Desktop/FIBERS/010726/siORC...
3,MGS1,siORC1_MGS1-03_Fiber_length,Fiber_length,68.244,0968-0321,/mnt/c/users/helen/Desktop/FIBERS/010726/siORC...
4,MGS1,siORC1_MGS1-03_Fiber_length,Fiber_length,100.000,0581-0161,/mnt/c/users/helen/Desktop/FIBERS/010726/siORC...


# Process replication speed

In [8]:
conversion_factor = 2.59 # kb/µm
time = 20 # minutes

In [10]:
# Checking speed file
counts = speed.groupby("File").size()
odd_files = counts[counts % 2 != 0].index.tolist()

if len(odd_files) == 0:
    print("All files contain an even number of fibers.")
else:
    print("The following files contain an odd number of fibers will be removed:")
    print(*odd_files, sep="\n")
    
    # Removing odd files from speed dataframe
    speed = speed[~speed["File"].isin(odd_files)].copy()

All files contain an even number of fibers.


In [18]:
# Add extra inedex to group pairs of files
speed["Index"] = speed.groupby("File").cumcount() // 2

# Calculate sum of fiber length in pairs
speed_processed = speed.groupby(["File", "Index"], as_index=False).agg(
        Total_Length=("Length", "sum"),
        ROI=("ROI", list),
        Path=("Path", "first"),
        Sample_name=("Sample_name", "first")
        )

# Convert speed to kb/min
speed_processed['Speed_kb_min'] = speed_processed['Total_Length'].apply(lambda x: x * conversion_factor / time)

# Delete extra columns
replication_speed = speed_processed[['Sample_name', 'File', 'Speed_kb_min', 'ROI', 'Path']]

# Info
#n_fibers = len(replication_speed)
#print(f"The amount of fibers is: {n_fibers}")

# Sample names
speed_sample_names = set(replication_speed['Sample_name'])
print(f"There are the following samples: {speed_sample_names}")

There are the following samples: {'WT', 'HaloEmpty', 'MGS5', 'MGS1', 'MGS4', 'MGS2', 'MGS3'}


In [19]:
replication_speed.head()

,Sample_name,File,Speed_kb_min,ROI,Path
0,WT,HCl_1h15min_15o-02_Fiber_length,7.268058,"[0172-0998, 0159-1032]",/mnt/c/users/helen/Desktop/FIBERS/WT_fibers_sp...
1,WT,HCl_1h15min_15o-06_Fiber_length,12.962302,"[0660-0135, 0624-0181]",/mnt/c/users/helen/Desktop/FIBERS/WT_fibers_sp...
2,WT,HCl_1h15min_15o-06_Fiber_length,9.868806,"[0627-0140, 0600-0174]",/mnt/c/users/helen/Desktop/FIBERS/WT_fibers_sp...
3,WT,HCl_1h15min_15o-06_Fiber_length,12.087918,"[0513-0309, 0472-0351]",/mnt/c/users/helen/Desktop/FIBERS/WT_fibers_sp...
4,WT,HCl_1h15min_15o-07_Fiber_length,8.567072,"[0490-0561, 0450-0575]",/mnt/c/users/helen/Desktop/FIBERS/WT_fibers_sp...


# Process IOD

In [ ]:
iod['IOD_kb'] = iod['Length'].apply(lambda x: x * conversion_factor)
iod_kb = iod[["Sample_name", "File", 'IOD_kb', 'ROI', 'Path']]

# Info
#n_origins = len(iod_kb)
#print(f"The amount of origins is: {n_origins}")

# Samples names
iod_sample_names = set(iod_kb['Sample_name'])
print(f"There are the following samples: {iod_sample_names}")

In [ ]:
set(iod_kb['Sample_name'])

# Graphs

## Replication speed graph

In [ ]:
plt.figure(figsize=(7, 5))

# Variables
data_plot = replication_speed
var = "Speed_kb_min"

# Order of groups (optional)
sample_order = ["WT", "MGS1", "MGS2", "MGS3", "MGS4", "MGS5"]

groups = []
labels = []

for sample in sample_order:
        values = data_plot.loc[
        data_plot["Sample_name"] == sample,
        var
    ]
        
        groups.append(values)
        labels.append(sample)

    

bp = plt.boxplot(
    groups,
    patch_artist=True,
    showfliers=False,
    widths=0.6,
)

for box in bp["boxes"]:
    box.set(facecolor="#4C72B0", alpha=0.6)

# Jittered dots
for i, values in enumerate(groups, start=1):
    x = np.random.normal(i, 0.05, len(values))
    plt.scatter(x, values, s=20, color="black", alpha=0.7, zorder=3)

plt.ylabel("Replication speed (kb/min)")
plt.xticks(range(1, len(labels) + 1), labels)
plt.grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.savefig(f"{input_dir}/replication_speed_boxplot.png", dpi=600, bbox_inches="tight")
print(f"Plot is saved in the directory: {input_dir}")

plt.show()

## IOD graph

In [ ]:
plt.figure(figsize=(7, 5))

# Variables
data_plot = iod_kb
var = "IOD_kb"

# Order of groups (optional)
sample_order = ["WT", "MGS1", "MGS2", "MGS3", "MGS4", "MGS5"]

groups = []
labels = []

for sample in sample_order:
        values = data_plot.loc[
        data_plot["Sample_name"] == sample,
        var
    ]
        
        groups.append(values)
        labels.append(sample)

    

bp = plt.boxplot(
    groups,
    patch_artist=True,
    showfliers=False,
    widths=0.6,
)

for box in bp["boxes"]:
    box.set(facecolor="#4C72B0", alpha=0.6)

# Jittered dots
for i, values in enumerate(groups, start=1):
    x = np.random.normal(i, 0.05, len(values))
    plt.scatter(x, values, s=20, color="black", alpha=0.7, zorder=3)

plt.ylabel("iod_kb")
plt.xticks(range(1, len(labels) + 1), labels)
plt.grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
#plt.savefig(f"{input_dir}/replication_speed_boxplot.png", dpi=600, bbox_inches="tight")
#print(f"Plot is saved in the directory: {input_dir}")

plt.show()

# Statistical analysis

## U-test

In [ ]:
# Data
data_plot = iod_kb
var = "IOD_kb"

wt = data_plot.loc[
    data_plot["Sample_name"] == "WT",
    var
].dropna()

results = []

for sample in ["MGS1", "MGS2", "MGS3", "MGS4", "MGS5"]:

    mutant = data_plot.loc[
        data_plot["Sample_name"] == sample,
        var
    ].dropna()

    stat, p = mannwhitneyu(
        wt,
        mutant
    )

    results.append({
        "Comparison": f"WT vs {sample}",
        "WT_n": len(wt),
        "Sample_n": len(mutant),
        "U": stat,
        "p-value": p,
    })

stats_u_df = pd.DataFrame(results)

print(stats_u_df)

## T-Test

In [ ]:
# Data
data_plot = iod_kb
var = "IOD_kb"

wt = data_plot.loc[
    data_plot["Sample_name"] == "WT",
    var
].dropna()

results = []

for sample in ["MGS1", "MGS2", "MGS3", "MGS4", "MGS5"]:

    mutant = data_plot.loc[
        data_plot["Sample_name"] == sample,
        var
    ].dropna()

    stat, p = ttest_ind(
        wt,
        mutant,
        equal_var=False,   # Welch's t-test (recommended)
    )

    results.append({
        "Comparison": f"WT vs {sample}",
        "WT_n": len(wt),
        "Sample_n": len(mutant),
        "U": stat,
        "p-value": p,
    })

stats_t_df = pd.DataFrame(results)

print(stats_t_df)

# Data export

In [ ]:

replication_speed.to_excel(f"{input_dir}/replication_speed.xlsx", index=False)
iod_kb.to_excel(f"{input_dir}/iod_kb.xlsx", index=False)